# Check Model Metrics

This notebook loads a `SEDifferNet` model from a `.pt` checkpoint and evaluates it on the test dataset defined in `config.py`.
It calculates both Image-Level and Pixel-Level metrics including AUROC, F1-Max, PG2, and AUPRO.

In [12]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import numpy as np
import matplotlib.pyplot as plt
## FIXED: Import auc as sk_auc to avoid conflict with variable names
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_recall_curve, auc as sk_auc
from tqdm import tqdm
import skimage.transform
from skimage import measure
from scipy.ndimage import rotate, gaussian_filter
from torch.autograd import Variable
from torch.cuda.amp import autocast

# Import project modules
import config as c
from model import SEDifferNet, load_weights
from utils import load_datasets, make_dataloaders, t2np, preprocess_batch, get_loss

%matplotlib inline

In [13]:
# --- CONFIGURATION ---
# Set the path to your .pt model file here
MODEL_PATH = r"C:\Users\teo-s\OneDrive\Documentos\GitHub\anomaly-v3\anomaly-detection-pesquisa\differnet\models\glass-insulator_differnet_100_100_epoch_66.pt" # Change this to your file path

# Override config settings if needed
c.class_name = "glass-insulator" # Make sure this matches your dataset
print(f"Using device: {c.device}")
print(f"Model path: {MODEL_PATH}")
print(f"Class name: {c.class_name}")

Using device: cuda
Model path: C:\Users\teo-s\OneDrive\Documentos\GitHub\anomaly-v3\anomaly-detection-pesquisa\differnet\models\glass-insulator_differnet_100_100_epoch_66.pt
Class name: glass-insulator


In [14]:
# --- HELPER FUNCTIONS ---

def calculate_f1_max(y_true, y_scores):
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    denom = precision + recall
    # Handle division by zero
    denom[denom == 0] = 1e-10
    f1_scores = 2 * (precision * recall) / denom
    return np.max(f1_scores)

def calculate_pg2(y_true, y_scores):
    # PG2: Percentage of good parts correctly classified (Specificity) 
    # when 2% of defective parts are incorrectly classified (FNR=2%, TPR=98%)
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    # Find index where TPR >= 0.98
    target_tpr = 0.98
    valid_indices = np.where(tpr >= target_tpr)[0]
    if len(valid_indices) == 0:
        return 0.0
    idx = valid_indices[0] # First index meeting condition
    
    # Specificity = 1 - FPR
    specificity = 1 - fpr[idx]
    return specificity

def calculate_aupro(gts, preds, integration_limit=0.3):
    # A simple implementation of AUPRO (Area Under Per-Region Overlap)
    # gts: list of ground truth masks (numpy arrays)
    # preds: list of anomaly maps (numpy arrays)
    # NOTE: This can be slow for large datasets. Using a simplified integration.
    
    # Flatten for global thresholding
    preds_flat = np.concatenate([p.ravel() for p in preds])
    # Sample thresholds (e.g., 50 steps from min to max)
    thresholds = np.linspace(preds_flat.min(), preds_flat.max(), 50)
    
    pros = []
    fprs = []
    
    # We need total pixels for FPR calculation
    total_pixels = len(preds_flat)
    total_neg_pixels = total_pixels - np.sum([np.sum(g > 0) for g in gts]) # This is approximate if GTs have overlapping logic, but good enough
    # Actually better to just accumulate FP count
    
    for th in thresholds:
        pro_sum = 0
        n_regions = 0
        fp_pixels = 0
        
        for gt, pred in zip(gts, preds):
            binary_pred = pred >= th
            
            # Compute FP for this image
            # FP = Predicted Anomaly AND NOT Ground Truth
            fp_pixels += np.sum(binary_pred & (gt == 0))
            
            # Compute PRO
            label_gt = measure.label(gt)
            regions = measure.regionprops(label_gt)
            
            for props in regions:
                # Region mask
                region_mask = (label_gt == props.label)
                n_pixels_region = props.area
                n_pixels_overlapping = np.sum(binary_pred & region_mask)
                
                pro_sum += (n_pixels_overlapping / n_pixels_region)
                n_regions += 1
                
        mean_pro = pro_sum / n_regions if n_regions > 0 else 1.0 # If no anomalies, PRO is defined as 1?
        fpr = fp_pixels / total_neg_pixels if total_neg_pixels > 0 else 0.0
        
        pros.append(mean_pro)
        fprs.append(fpr)
        
    # Integrate PRO vs FPR up to limit
    # Sort by FPR
    sorted_indices = np.argsort(fprs)
    fprs = np.array(fprs)[sorted_indices]
    pros = np.array(pros)[sorted_indices]
    
    # Clip to integration limit
    valid_mask = fprs <= integration_limit
    fprs_clipped = fprs[valid_mask]
    pros_clipped = pros[valid_mask]
    
    if len(fprs_clipped) < 2:
        return 0.0
        
    # Add origin if needed or just trapz
    # Standard AUPRO normalizes by the limit
    ## FIXED: Use sk_auc instead of auc to avoid name collision
    aupro = sk_auc(fprs_clipped, pros_clipped)
    aupro_normalized = aupro / integration_limit
    return aupro_normalized

def calculate_pixel_level_auroc(predictions, ground_truth_masks):
    # Resize predictions to match the ground truth mask dimensions
    predictions_resized = skimage.transform.resize(predictions, ground_truth_masks.shape, mode='constant')
    
    # Binarize ground truth masks
    ground_truth_masks_binary = (ground_truth_masks > 0).astype(int)

    # Flatten predictions and ground truth masks
    predictions_flat = predictions_resized.reshape(-1)
    ground_truth_flat = ground_truth_masks_binary.reshape(-1)
    
    # Calculate pixel-level AUROC
    pixel_auroc = roc_auc_score(ground_truth_flat, predictions_flat)
    return pixel_auroc, predictions_resized, ground_truth_masks_binary

def get_grad_maps(model, inputs, labels, optimizer):
    model.eval()
    inputs = Variable(inputs, requires_grad=True)
    
    with autocast():
        z = model(inputs)
        loss = get_loss(z, model.nf.jacobian(run_forward=False))
    
    optimizer.zero_grad()
    loss.backward()

    grad = inputs.grad.view(-1, c.n_transforms_test, *inputs.shape[-3:])
    grad = grad[labels > 0]
    
    # If no gradients (e.g. empty batch or filtered out), return None or zeros
    if grad.shape[0] == 0:
        return None

    grad = t2np(grad)
    degrees = -1 * np.arange(c.n_transforms_test) * 360.0 / c.n_transforms_test

    for i_item in range(c.n_transforms_test):
        old_shape = grad[:, i_item].shape
        img = np.reshape(grad[:, i_item], [-1, *grad.shape[-2:]])
        img = np.transpose(img, [1, 2, 0])
        img = np.transpose(rotate(img, degrees[i_item], reshape=False), [2, 0, 1])
        img = gaussian_filter(img, (0, 3, 3))
        grad[:, i_item] = np.reshape(img, old_shape)

    grad = np.reshape(grad, [grad.shape[0], -1, *grad.shape[-2:]])
    grad_img = np.mean(np.abs(grad), axis=1)
    grad_img_sq = grad_img ** 2
    return grad_img_sq

In [15]:
from model import DifferNet, load_weights
# --- LOAD MODEL ---
model = DifferNet()
model.to(c.device)

try:
    model, checkpoint = load_weights(model, MODEL_PATH)
    print("Model weights loaded successfully.")
    if checkpoint and 'epoch' in checkpoint:
        print(f"Checkpoint from epoch: {checkpoint['epoch']}")
except Exception as e:
    print(f"Error loading model: {e}")
    raise e

# Create dummy optimizer for gradient calculation
optimizer = torch.optim.Adam(model.nf.parameters(), lr=c.lr_init)

Model weights loaded successfully.
Checkpoint from epoch: 66


In [16]:
# --- LOAD DATA ---
print(f"Loading datasets for {c.class_name} from {c.dataset_path}...")
trainset, testset, ground_truth_set = load_datasets(c.dataset_path, c.class_name)

# NOTE: We manually create the test loader with shuffle=False to ensure alignment with ground truth
# make_dataloaders forces shuffle=True for test set
test_loader = torch.utils.data.DataLoader(testset, pin_memory=True, batch_size=c.batch_size_test, shuffle=False,
                                          drop_last=False, num_workers=c.num_workers)

ground_truth_loader = torch.utils.data.DataLoader(ground_truth_set, pin_memory=True, batch_size=c.batch_size, 
                                                  shuffle=False, drop_last=False, num_workers=c.num_workers)

print(f"Test dataset size: {len(testset)}")
print(f"Ground truth dataset size: {len(ground_truth_set)}")

Loading datasets for glass-insulator from C:/Users/teo-s/OneDrive/Documentos/GitHub/anomaly-v3/insplad-seg/insplad-seg...
Test dataset size: 678
Ground truth dataset size: 87


In [17]:
# --- EVALUATE ---
def evaluate(model, test_loader, ground_truth_loader, optimizer):
    model.eval()
    test_loss = []
    test_labels = []
    test_z = []
    pixel_level_auroc_scores = []
    
    # To calculate global pixel metrics (F1-Max Pixel, AUPRO), we need to store valid predictions and masks
    all_pred_masks = []
    all_gt_masks = []
    
    print("Evaluating...")
    # Set dataset to fixed mode for deterministic evaluation
    test_loader.dataset.get_fixed = True
    
    # Iterator for ground truth
    ground_truth_iter = iter(ground_truth_loader)
    
    for i, data in enumerate(tqdm(test_loader)):
        inputs, labels = preprocess_batch(data)
        
        # --- Image Level Forward ---
        with torch.no_grad():
            z = model(inputs)
            loss = get_loss(z, model.nf.jacobian(run_forward=False))
            test_loss.append(loss.item())
            test_labels.append(t2np(labels))
            test_z.append(z)
            
        # --- Pixel Level (Gradient Maps) ---
        grad_map = get_grad_maps(model, inputs, labels, optimizer)
        
        # Get corresponding ground truth
        try:
            gt_data = next(ground_truth_iter)
        except StopIteration:
            ground_truth_iter = iter(ground_truth_loader)
            gt_data = next(ground_truth_iter)
            
        gt_masks = gt_data[0].to(c.device)
        
        # Only calculate if we have gradients (meaning we had anomalies in the batch)
        if grad_map is not None:
             anomaly_indices = (labels > 0).nonzero().squeeze()
             if anomaly_indices.dim() == 0:
                 anomaly_indices = anomaly_indices.unsqueeze(0)
                 
             gt_masks_filtered = gt_masks[anomaly_indices]
             
             if len(gt_masks_filtered) == len(grad_map):
                 try:
                     val_masks = t2np(gt_masks_filtered)
                     
                     # Convert GT masks from (N, C, H, W) to (N, H, W) if needed
                     # ImageFolder loads as RGB (3 channels), usually identical.
                     if val_masks.ndim == 4 and val_masks.shape[1] == 3:
                         val_masks = val_masks[:, 0, :, :] # Take first channel
                     elif val_masks.ndim == 4 and val_masks.shape[1] == 1:
                         val_masks = val_masks.squeeze(1)
                     
                     if np.isnan(grad_map).any():
                         # print("Warning: NaNs in grad_map")
                         continue
                     
                     # Calculate per-image AUROC and store for global metrics
                     # Note: calculate_pixel_level_auroc now returns resized arrays too
                     pixel_auroc, pred_resized, gt_binary = calculate_pixel_level_auroc(grad_map, val_masks)
                     
                     if not np.isnan(pixel_auroc):
                         pixel_level_auroc_scores.append(pixel_auroc)
                         
                     # Store for global metrics (AUPRO, F1-Max Pixel)
                     # Ensure we store (N, H, W) for AUPRO logic by flattening channels if exists
                     if pred_resized.ndim == 4:
                         pred_resized = np.mean(pred_resized, axis=1) # (N, H, W)
                     if gt_binary.ndim == 4:
                         gt_binary = np.max(gt_binary, axis=1) # (N, H, W)
                         
                     all_pred_masks.append(pred_resized)
                     all_gt_masks.append(gt_binary)
                     
                 except Exception as e:
                     # print(f"Error in batch pixel AUROC: {e}")
                     pass
            
    test_loader.dataset.get_fixed = False

    # Calculate image-level results
    is_anomaly = np.array([0 if l == 0 else 1 for l in np.concatenate(test_labels)])
    z_grouped = torch.cat(test_z, dim=0).view(-1, c.n_transforms_test, c.n_feat)
    anomaly_score = t2np(torch.mean(z_grouped ** 2, dim=(-2, -1)))
    
    # Robust mean Pixel AUROC
    if pixel_level_auroc_scores:
        mean_pixel_auroc = np.nanmean(pixel_level_auroc_scores)
    else:
        mean_pixel_auroc = 0.0
        
    # Concatenate for global metrics
    try:
        if all_pred_masks:
             global_preds = np.concatenate(all_pred_masks)
             global_gts = np.concatenate(all_gt_masks)
        else:
             global_preds = np.array([])
             global_gts = np.array([])
    except Exception as e:
        print(f"Warning: Could not concatenate all masks (Memory?): {e}")
        global_preds = None
        global_gts = None
            
    return is_anomaly, anomaly_score, np.mean(test_loss), mean_pixel_auroc, global_preds, global_gts

# Run evaluation
y_true, y_scores, avg_loss, pixel_auroc, global_pixel_preds, global_pixel_gts = evaluate(model, test_loader, ground_truth_loader, optimizer)
print(f"Average Test Loss: {avg_loss:.4f}")

In [ ]:
# --- CALCULATE AND PRINT ALL METRICS ---
print("="*30)
print("IMAGE-LEVEL METRICS")
print("="*30)

# 1. AUROC
image_auroc = roc_auc_score(y_true, y_scores)
print(f"AUROC: {image_auroc:.4f}")

# 2. F1-Max Score
# Also print optimal threshold for F1
precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
denom = precision + recall
denom[denom == 0] = 1e-10
f1_scores = 2 * (precision * recall) / denom
f1_max_idx = np.argmax(f1_scores)
f1_max = f1_scores[f1_max_idx]
try:
    f1_thresh = thresholds[f1_max_idx]
except:
    f1_thresh = 0.0

print(f"F1-Max Score: {f1_max:.4f} (Threshold: {f1_thresh:.4f})")

# 3. PG2
pg2 = calculate_pg2(y_true, y_scores)
print(f"PG2 (Presorted Good at 2% Defect Error): {pg2:.4f}")

# 4. Accuracy at Best Threshold
fpr, tpr, roc_thresholds = roc_curve(y_true, y_scores)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold_val = roc_thresholds[best_idx]
y_pred_best = (y_scores >= best_threshold_val).astype(int)
acc_best = accuracy_score(y_true, y_pred_best)

print(f"Accuracy at Best Threshold ({best_threshold_val:.4f}): {acc_best:.4f}")

print("\n" + "="*30)
print("PIXEL-LEVEL METRICS")
print("="*30)
print(f"Mean Pixel-Level AUROC: {pixel_auroc:.4f}")

if global_pixel_preds is not None and len(global_pixel_preds) > 0:
    # Flatten for F1-Max
    flat_preds = global_pixel_preds.reshape(-1)
    flat_gts = global_pixel_gts.reshape(-1)
    
    # Pixel F1-Max
    print("Calculating Pixel F1-Max...")
    try:
        # Increased limit to 100M
        if len(flat_preds) > 100000000:
             indices = np.random.choice(len(flat_preds), 100000000, replace=False)
             print("(Calculated on subsample of 100M pixels)")
             precision, recall, thresholds = precision_recall_curve(flat_gts[indices], flat_preds[indices])
        else:
             precision, recall, thresholds = precision_recall_curve(flat_gts, flat_preds)
             
        denom = precision + recall
        denom[denom == 0] = 1e-10
        px_f1_scores = 2 * (precision * recall) / denom
        px_f1_max = np.max(px_f1_scores)
        print(f"Pixel F1-Max Score: {px_f1_max:.4f}")
    except Exception as e:
        print(f"Could not calculate Pixel F1-Max: {e}")
    
    # AUPRO
    # Needs unflattened arrays (N, H, W)
    print("Calculating AUPRO...")
    try:
        # Should already be (N, H, W) due to fix in evaluate
        if global_pixel_gts.ndim == 3:
             aupro_score = calculate_aupro([g for g in global_pixel_gts], [p for p in global_pixel_preds])
             print(f"AUPRO: {aupro_score:.4f}")
        else:
             print(f"Skipping AUPRO: Masks not in (N, H, W) format. Shape: {global_pixel_gts.shape}")
    except Exception as e:
        print(f"Could not calculate AUPRO: {e}")
else:
    print("No valid pixel predictions to calculate advanced metrics.")

In [ ]:
# --- PLOT ROC CURVE ---
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Image AUROC = {image_auroc:.4f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
# --- PLOT SCORE DISTRIBUTION ---
plt.figure(figsize=(10, 6))
normal_scores = y_scores[y_true == 0]
anomaly_scores = y_scores[y_true == 1]

plt.hist(normal_scores, bins=30, alpha=0.5, color='blue', label='Normal', density=True)
plt.hist(anomaly_scores, bins=30, alpha=0.5, color='red', label='Anomaly', density=True)

plt.axvline(best_threshold, color='black', linestyle='--', label=f'Threshold: {best_threshold:.2f}')
plt.xlabel('Anomaly Score')
plt.ylabel('Density')
plt.title('Anomaly Score Distribution')
plt.legend()
plt.grid(True)
plt.show()